# 🌲 02 — Drishti: Train GBT Delay Predictor (Spark MLlib + MLflow)

This is the **ML core of Drishti**.

**Why GBTRegressor:**
- Native Spark MLlib → distributed training, checks track's "Spark MLlib" technical hook
- Handles mixed categorical + numeric features via StringIndexer + VectorAssembler
- Beats linear models by ~30% on tabular data with non-linear interactions (e.g., fog × peak-hour)
- SHAP works natively on tree-based models → interpretable explanations for the demo
- CPU-native, no GPU needed

**MLflow integration:** every run is logged with params, metrics, the model artifact, and registered to Unity Catalog as `setu_rail.gold.setu_delay_predictor`.

**Target metrics:** MAE ≤ 12 min, RMSE ≤ 22 min.

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, when, isnan, isnull, coalesce

mlflow.set_registry_uri("databricks-uc")   # Unity Catalog model registry
EXPERIMENT = "/Shared/setu-rail-drishti"
mlflow.set_experiment(EXPERIMENT)
print(f"MLflow experiment: {EXPERIMENT}")

## Step 0 — Inspect the feature table to understand column names

In [0]:
# Load and inspect the ML-ready feature table
df = spark.table("setu_rail.gold.features_delay_ml")

print(f"Total rows available: {df.count():,}")
print(f"\nColumns in the table:")
df.printSchema()

print(f"\nFirst few rows:")
df.show(3, truncate=False)

## Step 1 — Data preparation with intelligent null handling

In [0]:
# List of NUMERIC features that should exist in the table
# These are the real features from 02_build_silver_enriched.ipynb
NUMERIC_FEATURES = [
    "stop_seq",                   # 0-indexed stop sequence
    "total_stops",                # count of stops
    "cumulative_travel_min",      # minutes from origin departure
    "dwell_min",                  # arrival to departure time at station
    "scheduled_hour",             # hour of departure (0-23)
    "is_peak_hour",               # 1 if peak, 0 otherwise
    "pm25",                       # air quality proxy
    "no2",                        # another air quality metric
    "journey_day",                # day of multi-day journey
]

CATEGORICAL_FEATURES = [
    "train_no",
    "station_code",
    "train_type",
    "zone",
]

LABEL = "arrival_delay_min"

# Check which columns exist
existing_cols = set(df.columns)
print("Checking feature availability:")
print(f"Numeric features available: {[f for f in NUMERIC_FEATURES if f in existing_cols]}")
print(f"Categorical features available: {[f for f in CATEGORICAL_FEATURES if f in existing_cols]}")
print(f"Label available: {LABEL in existing_cols}")

# Filter to only features that exist
numeric_features_avail = [f for f in NUMERIC_FEATURES if f in existing_cols]
categorical_features_avail = [f for f in CATEGORICAL_FEATURES if f in existing_cols]

print(f"\nWill use {len(numeric_features_avail)} numeric + {len(categorical_features_avail)} categorical features")

In [0]:
# Prepare data: drop rows with null label, fill nulls in features with sensible defaults
df_clean = df.filter(col(LABEL).isNotNull())

# Fill numeric features with column median or defaults
fill_dict = {
    "stop_seq": 0,
    "total_stops": 50,
    "cumulative_travel_min": 0,
    "dwell_min": 5,
    "scheduled_hour": 12,
    "is_peak_hour": 0,
    "pm25": 60.0,
    "no2": 25.0,
    "journey_day": 1,
    "latitude": 20.0,
    "longitude": 77.0,
}

# Only fill columns that exist
actual_fill = {k: v for k, v in fill_dict.items() if k in df_clean.columns}
df_clean = df_clean.fillna(actual_fill)

# For categorical, fill with 'UNKNOWN'
cat_cols_fill = {f: "UNKNOWN" for f in categorical_features_avail if f in df_clean.columns}
df_clean = df_clean.fillna(cat_cols_fill)

print(f"Rows after cleaning: {df_clean.count():,}")
df_clean.show(2, truncate=False)

## Step 2 — Time-respecting train/test split

In [0]:
# If we have a month column, split by month (temporal integrity)
if "month" in df_clean.columns:
    train_df = df_clean.filter(col("month") <= 9)
    test_df  = df_clean.filter(col("month") > 9)
    print(f"Split by month: train (1-9) vs test (10-12)")
else:
    # Otherwise use random 80/20
    train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
    print(f"Random 80/20 split (no time column)")

print(f"Train rows: {train_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

## Step 3 — Build and train the ML Pipeline

In [0]:
# Create StringIndexers for categorical features
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx",
                          handleInvalid="keep")
            for c in categorical_features_avail]

# Build the feature vector
indexed_cols = [f"{c}_idx" for c in categorical_features_avail] + numeric_features_avail

assembler = VectorAssembler(
    inputCols=indexed_cols,
    outputCol="features",
    handleInvalid="skip",
)

print(f"Feature vector will have {len(indexed_cols)} inputs:")
print(f"  Categorical (indexed): {[f'{c}_idx' for c in categorical_features_avail]}")
print(f"  Numeric: {numeric_features_avail}")

In [0]:
# GBT model — CPU-friendly parameters for Free Edition
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=LABEL,
    maxIter=50,           # moderate iterations
    maxDepth=6,           # shallow trees
    stepSize=0.08,        # conservative learning rate
    maxBins=128,          # reasonable for Free Edition
    seed=42,
    subsamplingRate=0.8,  # help with memory
)

# Assemble pipeline
pipeline = Pipeline(stages=indexers + [assembler, gbt])

print("✅ Pipeline created with:")
print(f"   - {len(indexers)} StringIndexers")
print(f"   - 1 VectorAssembler")
print(f"   - 1 GBTRegressor")

## Step 4 — Train and log to MLflow

In [0]:
# Train with MLflow tracking
with mlflow.start_run(run_name="gbt_v1_baseline") as run:
    print("🚀 Training GBT model...")
    model = pipeline.fit(train_df)
    print("✅ Training complete")

    # Predictions
    print("📊 Evaluating on test set...")
    preds = model.transform(test_df)

    # Metrics
    mae  = RegressionEvaluator(labelCol=LABEL, metricName="mae").evaluate(preds)
    rmse = RegressionEvaluator(labelCol=LABEL, metricName="rmse").evaluate(preds)
    r2   = RegressionEvaluator(labelCol=LABEL, metricName="r2").evaluate(preds)

    print(f"\n📈 Metrics:")
    print(f"   MAE:  {mae:.2f} min")
    print(f"   RMSE: {rmse:.2f} min")
    print(f"   R²:   {r2:.3f}")

    # Log parameters
    mlflow.log_params({
        "maxIter": 50,
        "maxDepth": 6,
        "stepSize": 0.08,
        "maxBins": 128,
        "train_rows": train_df.count(),
        "test_rows": test_df.count(),
        "num_features": len(indexed_cols),
    })

    # Log metrics
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

    # Log model
    print("\n💾 Logging model to MLflow...")
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="setu_gbt_model",
        registered_model_name="setu_rail.gold.setu_delay_predictor",
    )

    print(f"✅ Run ID: {run.info.run_id}")
    print(f"✅ Model registered to UC as setu_rail.gold.setu_delay_predictor")

## Step 5 — Baseline comparison

In [0]:
# What if we just always predicted the mean?
from pyspark.sql.functions import lit, abs as abs_

mean_delay = train_df.agg({LABEL: "avg"}).collect()[0][0]
print(f"Mean delay in training set: {mean_delay:.2f} min")

# Compute baseline MAE
base_preds = test_df.withColumn("prediction", lit(mean_delay))
baseline_mae = base_preds.withColumn("error", abs_(col(LABEL) - col("prediction"))) \
                         .agg({"error": "avg"}).collect()[0][0]

improvement = 100 * (baseline_mae - mae) / baseline_mae
print(f"\nBaseline (always-predict-mean) MAE: {baseline_mae:.2f} min")
print(f"GBT model MAE:                       {mae:.2f} min")
print(f"🎯 Improvement:                      {improvement:.1f}%")

## Step 6 — Save predictions for dashboard

In [0]:
# Select key columns for the dashboard
cols_to_keep = ["train_no", "station_code", "scheduled_hour",
                 col(LABEL).alias("actual_delay_min"),
                 col("prediction").alias("predicted_delay_min")]

# Add optional columns if they exist
if "pm25" in preds.columns:
    cols_to_keep.append("pm25")
if "no2" in preds.columns:
    cols_to_keep.append("no2")
if "zone" in preds.columns:
    cols_to_keep.append("zone")

predictions_daily = preds.select(cols_to_keep)

(predictions_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.gold.predictions_daily"))

print(f"✅ Saved {predictions_daily.count():,} predictions to gold.predictions_daily")

## Step 7 — Create production alias in MLflow

## Summary

✅ **Model trained & logged to UC**
- Metrics: MAE {mae:.2f}min, RMSE {rmse:.2f}min
- Used: {len(categorical_features_avail)} categorical + {len(numeric_features_avail)} numeric features
- Distributed via Spark MLlib + Delta partitioning

✅ **Next:** `03_shap_explainability.ipynb`